# AccessAI v5.2 - Prototipo de detección urbana

Este notebook documenta el flujo experimental de detección urbana con YOLO y el ROD-Dataset.

Objetivo: mantener una pipeline reproducible con 4 clases finales agrupadas: `Obstaculo_Dinamico`, `Obstaculo_Fijo`, `Barrera_Arquitectonica` e `Infraestructura_Peatonal`.

> Es un prototipo técnico experimental. Las detecciones no deben interpretarse como una validación final de accesibilidad urbana sin un dataset específico y métricas adecuadas.

## Outline v5.2

# AccessAI v5.2 - Prototipo de detección urbana

**Clases finales:** `Obstaculo_Dinamico`, `Obstaculo_Fijo`, `Barrera_Arquitectonica` e `Infraestructura_Peatonal`.

1. [01. Entorno y GPU](#01-entorno-y-gpu)
2. [02. Comprobar Ultralytics](#02-comprobar-ultralytics)
3. [03. Configuración del dataset](#03-configuración-del-dataset)
4. [04. Mapeo de clases del ROD-Dataset](#04-mapeo-de-clases-del-rod-dataset)
5. [05. Crear `data_filtrado.yaml`](#05-crear-data_filtradoyaml)
6. [06. Preparar etiquetas](#06-preparar-etiquetas)
7. [07. Comprobar estructura del dataset](#07-comprobar-estructura-del-dataset)
8. [08. Entrenamiento YOLO26X](#entrenamiento)
9. [09. Cargar el `best.pt` entrenado](#09-cargar-el-bestpt-entrenado)
10. [10. Evaluación en el conjunto test](#10-evaluación-en-el-conjunto-test)
11. [11. Cargar `results.csv` y visualizar curvas](#11-cargar-resultscsv-y-visualizar-curvas)
12. [12. Seleccionar una imagen de prueba](#12-seleccionar-una-imagen-de-prueba)
13. [13. Inferencia](#13-inferencia)
14. [14. Guardar y mostrar la imagen anotada](#14-guardar-y-mostrar-la-imagen-anotada)
15. [15. Resumen del experimento](#15-resumen-del-experimento)

In [1]:
from pathlib import Path
import sys

print('=' * 80)
print('ACCESSAI - ENTORNO Y GPU')
print('=' * 80)

print('Python:', sys.executable)

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
    if torch.cuda.is_available():
        device = 0
        print('GPU:', torch.cuda.get_device_name(device))
        props = torch.cuda.get_device_properties(device)
        print(f'VRAM total: {props.total_memory / 1024**3:.2f} GB')
        torch.set_float32_matmul_precision('high')
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
except ModuleNotFoundError:
    print('Faltan dependencias: activa el entorno del proyecto e instala ultralytics y torch.')
    raise

# ============================================================
# ACCESSAI - 01. ENTORNO Y GPU
# ============================================================

from pathlib import Path
import os
import torch

print("=" * 70)
print("ACCESSAI - ENTORNO")
print("=" * 70)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"CUDA de PyTorch: {torch.version.cuda}")
print(f"HIP: {torch.version.hip}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

    props = torch.cuda.get_device_properties(0)

    print(
        f"Memoria GPU: "
        f"{props.total_memory / 1024**3:.2f} GB"
    )

    print(
        f"Compute capability: "
        f"{props.major}.{props.minor}"
    )

    print(
        f"Memoria reservada actualmente: "
        f"{torch.cuda.memory_reserved(0) / 1024**3:.2f} GB"
    )

    print(
        f"Memoria asignada actualmente: "
        f"{torch.cuda.memory_allocated(0) / 1024**3:.2f} GB"
    )
else:
    print("No hay GPU disponible.")

print("=" * 70)


# [markdown]
# # 02. Comprobar Ultralytics

# ============================================================
# ACCESSAI - 02. ULTRALYTICS
# ============================================================

import ultralytics

print("Ultralytics:", ultralytics.__version__)

!python -m pip show ultralytics

ACCESSAI - ENTORNO Y GPU
Python: c:\Users\Usuario\.conda\envs\proyecto_yolo\python.exe
PyTorch: 2.14.0+cu126
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total: 8.00 GB
ACCESSAI - ENTORNO
PyTorch: 2.14.0+cu126
CUDA disponible: True
CUDA de PyTorch: 12.6
HIP: None
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU count: 1
Memoria GPU: 8.00 GB
Compute capability: 8.9
Memoria reservada actualmente: 0.00 GB
Memoria asignada actualmente: 0.00 GB
Ultralytics: 8.4.69
Name: ultralytics
Version: 8.4.69
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: c:\Users\Usuario\.conda\envs\proyecto_yolo\Lib\site-packages
Requires: matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, re

In [2]:
try:
    import ultralytics
    print('Ultralytics:', ultralytics.__version__)
except ModuleNotFoundError:
    print('Ultralytics no está instalado.')
    raise

Ultralytics: 8.4.69


In [3]:
def encontrar_raiz_proyecto():
    actual = Path.cwd().resolve()
    candidatos = [actual] + list(actual.parents)
    for candidato in candidatos:
        dataset = candidato / 'DATA' / 'ROD-Dataset' / 'dataset'
        if dataset.exists():
            return candidato
    return actual

ROOT = encontrar_raiz_proyecto()
DATASET_PATH = ROOT / 'DATA' / 'ROD-Dataset' / 'dataset'
print('ROOT:', ROOT)
print('DATASET_PATH:', DATASET_PATH)
print('Existe dataset:', DATASET_PATH.exists())

yaml_original = DATASET_PATH / 'data.yaml'
yaml_filtrado = DATASET_PATH / 'data_filtrado.yaml'
print('data.yaml:', yaml_original.exists())
print('data_filtrado.yaml:', yaml_filtrado.exists())

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'No existe el dataset: {DATASET_PATH}')

DATA_YAML = yaml_filtrado if yaml_filtrado.exists() else yaml_original
print('YAML usado:', DATA_YAML)

ROOT: C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2
DATASET_PATH: C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset
Existe dataset: True
data.yaml: True
data_filtrado.yaml: True
YAML usado: C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml


## Mapeo de clases

El dataset original tiene 25 clases; para este prototipo se remapean a 4 categorías experimentales:

- `Obstaculo_Dinamico`
- `Obstaculo_Fijo`
- `Barrera_Arquitectonica`
- `Infraestructura_Peatonal`

In [4]:
CLASES_ORIGINALES = [
    'Bike',
    'Building',
    'Car',
    'Person',
    'Stairs',
    'Traffic sign',
    'Electrical Pole',
    'Road',
    'Motorcycle',
    'Dustbin',
    'Dog',
    'Manhole',
    'Tree',
    'Guard rail',
    'Pedestrian crosswalk',
    'Truck',
    'Bus',
    'Bench',
    'Traffic Cone',
    'Fire hydrant',
    'Teraffic Barrel',
    'Plant Pot',
    'Electrical Box',
    'Chair',
    'Bicycle Rack',
]

MAPEO_CLASES = {
    0: 0,  # Bike
    2: 0,  # Car
    3: 0,  # Person
    8: 0,  # Motorcycle
    10: 0, # Dog
    15: 0, # Truck
    16: 0, # Bus
    5: 1,  # Traffic sign
    6: 1,  # Electrical Pole
    9: 1,  # Dustbin
    12: 1, # Tree
    13: 1, # Guard rail
    17: 1, # Bench
    18: 1, # Traffic Cone
    19: 1, # Fire hydrant
    20: 1, # Teraffic Barrel
    21: 1, # Plant Pot
    22: 1, # Electrical Box
    23: 1, # Chair
    24: 1, # Bicycle Rack
    4: 2,  # Stairs
    7: 3,  # Road
    11: 3, # Manhole
    14: 3, # Pedestrian crosswalk
}

NUEVOS_NOMBRES = [
    'Obstaculo_Dinamico',
    'Obstaculo_Fijo',
    'Barrera_Arquitectonica',
    'Infraestructura_Peatonal',
]

print('Clases finales:', NUEVOS_NOMBRES)
print('Número de clases:', len(NUEVOS_NOMBRES))

Clases finales: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']
Número de clases: 4


In [5]:
import yaml
from collections import Counter

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Configuración del dataset:')
print(cfg)
print('Clase objetivo / nc:', cfg.get('nc'))
print('Nombres:', cfg.get('names'))

Configuración del dataset:
{'path': 'C:\\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\\proyecto2\\DATA\\ROD-Dataset\\dataset', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 4, 'names': ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']}
Clase objetivo / nc: 4
Nombres: ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']


## Entrenamiento

Este bloque usa `yolo26x.pt` como base y entrena el modelo sobre el dataset filtrado con 4 clases. En entornos con GPU CUDA, conviene usar `batch=-1` para AutoBatch y controlar el uso de VRAM.

## Diagnostico y ajustes para RTX 4060 Laptop (8 GB)

### Diagnostico del intento anterior

El registro disponible confirma que Ultralytics 8.4.69 cargó `yolo26x.pt`, transfirió 1080/1092 pesos, detectó `CUDA:0` como NVIDIA GeForce RTX 4060 Laptop GPU y pasó las comprobaciones AMP. El proceso llegó a 547/2399 batches del primer epoch y terminó por `KeyboardInterrupt`; no fue un `CUDA out of memory`. El progreso mostraba 3,8 s/iteracion, pero ese dato aislado no permite atribuir el cuello de botella a GPU, CPU o almacenamiento porque no se guardaron mediciones simultáneas. Ultralytics mostró `GPU_mem=8.71G` aunque PyTorch reporta 8 GB: contrástalo con la memoria dedicada y compartida de `nvidia-smi`/Administrador de tareas antes de volver a usar batch 8.

### Decisiones de configuración

| Perfil | Batch | Uso recomendado |
|---|---:|---|
| Seguro | 1 | Primer intento si el preflight con auto-batch falla o la memoria dedicada queda casi llena. |
| Recomendado | -1 | Auto-batch de Ultralytics, que estima el batch usando la GPU disponible. Es el perfil inicial de esta copia, no una garantía contra OOM. |
| Agresivo | 4 | Probar solo después de un preflight correcto con auto-batch y con memoria dedicada libre; no hay benchmark que garantice que sea más rápido. |

No repetiría `batch=8`: el registro ya alcanzó 8,71 GB de memoria GPU reportada. Auto-batch puede calcular una estimación distinta en cada inicio según procesos competidores; el preflight de 2% ayuda a detectar problemas de arranque, pero no garantiza que todas las iteraciones futuras quepan. Si falla, reinicia el kernel y usa `PROFILE = "safe"`.

- **AMP:** `amp=True` se conserva; el chequeo AMP ya pasó y reduce memoria/trabajo en hardware compatible.
- **Optimizador:** `optimizer="auto"` se conserva. En el intento anterior eligió AdamW con sus propios valores de learning rate y momentum, por lo que los valores manuales `lr0`/`momentum` se ignorarían.
- **Épocas y parada:** `epochs=20`; `patience=0` desactiva early stopping para que no termine antes. La validación sigue activa y se guardan `best.pt` y `last.pt`.
- **Aumentación:** se mantienen los defaults de detección de Ultralytics; no fuerzo aumentos adicionales sin evidencia de que el dataset los necesite. `close_mosaic=10` cierra mosaic en los últimos 10 epochs.
- **Guardado y plots:** `save_period=-1` evita checkpoints por epoch, pero se conservan `best.pt` y `last.pt`. `plots=True` genera gráficas de resultados; apagarlo puede ahorrar algo de trabajo al final, no acelera sustancialmente cada batch.
- **Cache:** `cache="disk"` usa espacio en disco, no VRAM. El intento anterior informó 19,1 GB para train y 3,4 GB para validación. El código lo activa solo si hay al menos 30 GB libres. `cache=True` puede requerir mucha RAM para imágenes decodificadas y no se recomienda sin conocer la RAM disponible. `cache=False` evita ocupar disco; úsalo si el almacenamiento es lento o está justo.
- **Workers:** esta notebook usa `workers=0` para evitar problemas de multiprocessing `spawn` en Jupyter/Windows. En un `.py` con bloque `if __name__ == "__main__":` usa hasta 4; si el DataLoader limita la GPU, compara 2 y 4. Más workers no siempre ayudan y aumentan uso de RAM/CPU.
- **Reproducibilidad:** se mantiene `seed=42`; `deterministic=False` da reproducibilidad razonable, no idéntica bit a bit, y evita algunas restricciones de determinismo.
- **Dispositivo y pesos:** `device=0` y `yolo26x.pt` mantienen la RTX 4060 y el transfer learning. La celda verifica CUDA y que el YAML contenga exactamente cuatro clases antes de iniciar.

El preflight configurado prueba un epoch sobre el 2% del entrenamiento sin validación ni checkpoints. No cuenta como resultado del modelo ni como benchmark; sirve para detectar errores de ruta, carga y memoria antes del trabajo completo.

### Notebook frente a script

En Jupyter, Windows crea procesos mediante `spawn`; workers mayores que cero pueden causar bloqueos, procesos duplicados o errores de serialización. Por eso la celda elige cero workers dentro del kernel. Para medir el máximo rendimiento del DataLoader, guarda el mismo entrenamiento como `.py`, conserva el bloque `if __name__ == "__main__":` y prueba `WORKERS=2` y después `4`. No ejecutes simultáneamente entrenamientos duplicados desde notebook y script.

### Monitorización en PowerShell

Abre otra ventana de PowerShell. El muestreo de GPU se actualiza cada segundo; termina con `Ctrl+C`:

```powershell
nvidia-smi --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw,power.limit,clocks.current.sm --format=csv -l 1
```

CPU, memoria disponible y cola/transferencia de disco (60 muestras, una por segundo):

```powershell
Get-Counter '\Processor(_Total)\% Processor Time','\Memory\Available MBytes','\PhysicalDisk(_Total)\Avg. Disk Queue Length','\PhysicalDisk(_Total)\Disk Bytes/sec' -SampleInterval 1 -MaxSamples 60
```

Si `nvidia-smi` no acepta un campo de reloj en ese driver, quita `clocks.current.sm` de la consulta. Si `Get-Counter` no reconoce los nombres por el idioma de Windows, usa el Administrador de tareas (Rendimiento: GPU, CPU, memoria y disco) o consulta los nombres locales con `Get-Counter -ListSet *`.

Interpreta varias muestras después del calentamiento, no un único instante. GPU sostenida alta suele indicar carga de cómputo; uso bajo persistente junto con CPU/disco ocupados sugiere que el DataLoader puede estar esperando. No existe un umbral universal: contrasta el tiempo por iteración, `GPU-Util`, VRAM dedicada y memoria compartida. VRAM dedicada cerca del límite y memoria compartida creciente elevan el riesgo de OOM y paginación. Temperatura y potencia dependen del TGP y del perfil térmico de esta laptop; compara `power.draw` con `power.limit` y busca caída de clocks con temperatura alta, no un umbral inventado.

### Resolucion de problemas

- **CUDA OOM:** reinicia el kernel para soltar memoria retenida; cierra otras aplicaciones que usen GPU; usa `PROFILE = "safe"` (batch 1). Conserva YOLO26X, 640 px, cuatro clases y 20 epochs. Si el OOM solo aparece en el entrenamiento completo, revisa también memoria compartida y otros procesos.
- **GPU utilization baja:** deja terminar el calentamiento inicial y observa varias iteraciones. Si CPU o disco están activos y la GPU espera, mueve el dataset/cache a SSD local y, en `.py`, prueba workers 2 y 4. En Jupyter permanece en 0 por estabilidad de Windows.
- **DataLoader lento:** confirma que el dataset está en almacenamiento local rápido, revisa espacio libre para cache de disco y evita antivirus/indexación sobre la carpeta del dataset si la política del equipo lo permite. Compara `cache=False` con `cache="disk"` en ejecuciones separadas; no se afirma de antemano cuál será más rápida.
- **VRAM casi llena:** busca memoria GPU dedicada en Administrador de tareas y procesos con `nvidia-smi`; usa batch 1 o auto-batch. `torch.cuda.empty_cache()` solo libera memoria cacheada no referenciada por PyTorch, no la que usan otros procesos.
- **Parece entrenar en CPU:** la celda aborta si `torch.cuda.is_available()` es falso; al inicio deben aparecer la RTX 4060 y `CUDA:0`, y Ultralytics debe registrar `device=0`. Si no, verifica el intérprete/entorno, `torch.version.cuda`, el driver y que no exista otro Python ejecutando una copia distinta.

No se ha ejecutado el entrenamiento en esta edición. No se reportan tiempos, utilización ni métricas nuevas; solo el intento previo descrito arriba.

In [ ]:
# ============================================================
# ACCESSAI - YOLO26X
# RTX 4060 LAPTOP GPU 8 GB
# 4 CLASES
#
# CONFIGURACIÓN ESTABLE PARA 8 GB VRAM
# batch=1
# imgsz=640
# AMP=True
# AdamW
# 20 epochs
# ============================================================

# ------------------------------------------------------------
# 0. CONFIGURACIÓN DE CUDA ANTES DE IMPORTAR PYTORCH
# ------------------------------------------------------------

import os

# Ayuda a reducir problemas de fragmentación de memoria CUDA.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ------------------------------------------------------------
# 1. IMPORTACIONES
# ------------------------------------------------------------

from datetime import datetime
from pathlib import Path
import sys
import shutil

import torch
import yaml
import ultralytics
from ultralytics import YOLO


# ------------------------------------------------------------
# 2. ROOT DEL PROYECTO
# ------------------------------------------------------------

if "__file__" in globals():
    ROOT = Path(__file__).resolve().parent
else:
    ROOT = Path(globals().get("ROOT", Path.cwd())).resolve()


# ------------------------------------------------------------
# 3. RUTAS
# ------------------------------------------------------------

DATA_YAML = Path(
    globals().get(
        "DATA_YAML",
        ROOT
        / "DATA"
        / "ROD-Dataset"
        / "dataset"
        / "data_filtrado.yaml",
    )
).resolve()

MODEL_PATH = (ROOT / "yolo26x.pt").resolve()


# ------------------------------------------------------------
# 4. CONFIGURACIÓN DE ENTRENAMIENTO
# ------------------------------------------------------------

MODEL_NAME = "YOLO26X"

EPOCHS = 20
IMGSZ = 640

# IMPORTANTE:
# No usar -1 porque activa AutoBatch.
# En 8 GB para YOLO26X comenzamos con batch 1.
BATCH = 1

DEVICE = 0

AMP = True

SEED = 42

DETERMINISTIC = False

# 0 = desactiva EarlyStopping.
# Las 20 épocas se ejecutarán completas.
PATIENCE = 0

# Cierra Mosaic durante las últimas 10 épocas.
CLOSE_MOSAIC = 10

PLOTS = True

SAVE_PERIOD = -1

# ------------------------------------------------------------
# Preflight
# ------------------------------------------------------------
#
# Se mantiene activado, pero TAMBIÉN usa batch=1.
# Así no ejecuta AutoBatch.
#
RUN_PREFLIGHT = True


# ------------------------------------------------------------
# 5. WORKERS
# ------------------------------------------------------------

IN_NOTEBOOK = "ipykernel" in sys.modules

if IN_NOTEBOOK:
    WORKERS = 0
else:
    WORKERS = min(4, os.cpu_count() or 1)


# ------------------------------------------------------------
# 6. CACHE
# ------------------------------------------------------------

CACHE = "disk"

if DATA_YAML.exists():

    FREE_DISK_GB = (
        shutil.disk_usage(DATA_YAML.parent).free
        / 1024**3
    )

    if FREE_DISK_GB < 30:

        print(
            f"Aviso: quedan {FREE_DISK_GB:.1f} GB libres."
        )

        print("Cache de dataset desactivada.")

        CACHE = False


# ------------------------------------------------------------
# 7. FUNCIÓN PARA LIBERAR CUDA
# ------------------------------------------------------------

def limpiar_cuda():

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ------------------------------------------------------------
# 8. FUNCIÓN PRINCIPAL
# ------------------------------------------------------------

def main():

    # --------------------------------------------------------
    # VALIDAR DATASET
    # --------------------------------------------------------

    if not DATA_YAML.is_file():

        raise FileNotFoundError(
            f"No existe el archivo:\n{DATA_YAML}\n\n"
            "Ejecuta primero la preparación del dataset "
            "que genera data_filtrado.yaml."
        )


    # --------------------------------------------------------
    # VALIDAR MODELO
    # --------------------------------------------------------

    if not MODEL_PATH.is_file():

        raise FileNotFoundError(
            f"No existe el modelo:\n{MODEL_PATH}\n\n"
            "Coloca yolo26x.pt en la raíz del proyecto."
        )


    # --------------------------------------------------------
    # LEER YAML
    # --------------------------------------------------------

    with DATA_YAML.open(
        "r",
        encoding="utf-8"
    ) as file:

        dataset_cfg = yaml.safe_load(file)


    class_names = dataset_cfg.get(
        "names",
        []
    )

    nc = dataset_cfg.get(
        "nc"
    )


    # --------------------------------------------------------
    # VALIDAR LAS 4 CLASES
    # --------------------------------------------------------

    if nc != 4 or len(class_names) != 4:

        raise ValueError(
            "Se esperaban exactamente 4 clases.\n"
            f"nc = {nc}\n"
            f"names = {class_names}"
        )


    # --------------------------------------------------------
    # VALIDAR CUDA
    # --------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA no está disponible.\n"
            "El entrenamiento se cancela para evitar utilizar CPU."
        )


    # --------------------------------------------------------
    # INFORMACIÓN GPU
    # --------------------------------------------------------

    gpu_name = torch.cuda.get_device_name(
        DEVICE
    )

    free_bytes, total_bytes = (
        torch.cuda.mem_get_info(DEVICE)
    )


    free_vram_gb = (
        free_bytes / 1024**3
    )

    total_vram_gb = (
        total_bytes / 1024**3
    )


    # --------------------------------------------------------
    # LIMPIAR CUDA ANTES DE ENTRENAR
    # --------------------------------------------------------

    limpiar_cuda()


    # --------------------------------------------------------
    # NOMBRE DEL RUN
    # --------------------------------------------------------

    run_name = (
        f"AccessAI_YOLO26X_4clases_"
        f"{datetime.now():%Y-%m-%d_%H-%M-%S}"
    )


    # --------------------------------------------------------
    # INFORMACIÓN
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("ACCESSAI - YOLO26X - ENTRENAMIENTO GPU")
    print("=" * 75)

    print(
        f"Ultralytics       : {ultralytics.__version__}"
    )

    print(
        f"PyTorch           : {torch.__version__}"
    )

    print(
        f"CUDA PyTorch      : {torch.version.cuda}"
    )

    print(
        f"CUDA disponible   : {torch.cuda.is_available()}"
    )

    print(
        f"GPU               : {gpu_name}"
    )

    print(
        f"VRAM libre/total  : "
        f"{free_vram_gb:.2f} / "
        f"{total_vram_gb:.2f} GB"
    )

    print(
        f"Modelo            : {MODEL_PATH}"
    )

    print(
        f"Dataset           : {DATA_YAML}"
    )

    print(
        f"Clases (4)        : {class_names}"
    )

    print(
        f"Batch             : {BATCH}"
    )

    print(
        f"Epochs            : {EPOCHS}"
    )

    print(
        f"Image size        : {IMGSZ}"
    )

    print(
        f"Device            : {DEVICE}"
    )

    print(
        f"AMP               : {AMP}"
    )

    print(
        f"Optimizer         : AdamW"
    )

    print(
        f"Workers           : "
        f"{WORKERS} "
        f"({'Jupyter' if IN_NOTEBOOK else 'script'})"
    )

    print(
        f"Cache             : {CACHE}"
    )

    print(
        f"Patience          : {PATIENCE}"
    )

    print(
        f"Close Mosaic      : {CLOSE_MOSAIC}"
    )

    print(
        f"Preflight         : {RUN_PREFLIGHT}"
    )

    print("=" * 75)


    # --------------------------------------------------------
    # 9. PREFLIGHT
    # --------------------------------------------------------

    if RUN_PREFLIGHT:

        print()
        print("=" * 75)
        print(
            "PREFLIGHT - 1 EPOCH / 2% DATASET / BATCH 1"
        )
        print("=" * 75)

        limpiar_cuda()

        preflight_model = YOLO(
            str(MODEL_PATH)
        )

        try:

            preflight_model.train(

                data=str(DATA_YAML),

                epochs=1,

                fraction=0.02,

                imgsz=IMGSZ,

                # MUY IMPORTANTE:
                # batch fijo.
                batch=1,

                device=DEVICE,

                amp=AMP,

                workers=WORKERS,

                cache=CACHE,

                optimizer="AdamW",

                seed=SEED,

                deterministic=DETERMINISTIC,

                val=False,

                save=False,

                plots=False,

                project=str(
                    ROOT / "runs" / "preflight"
                ),

                name=run_name,

                exist_ok=True,

                verbose=True,
            )


        except torch.cuda.OutOfMemoryError as exc:

            limpiar_cuda()

            raise RuntimeError(
                "\nCUDA OOM durante PREFLIGHT.\n\n"
                "El YOLO26X no cabe con batch=1 a "
                f"imgsz={IMGSZ} en la VRAM disponible.\n\n"
                "Siguiente configuración recomendada:\n"
                "  IMGSZ = 512\n"
                "  BATCH = 1\n"
                "  AMP = True\n"
            ) from exc


        finally:

            del preflight_model

            limpiar_cuda()


        print()
        print(
            "Preflight completado correctamente."
        )

        print(
            "El entrenamiento real comenzará "
            "desde yolo26x.pt."
        )


    # --------------------------------------------------------
    # 10. LIMPIEZA FINAL ANTES DEL TRAIN
    # --------------------------------------------------------

    limpiar_cuda()


    # --------------------------------------------------------
    # 11. MODELO REAL
    # --------------------------------------------------------

    model = YOLO(
        str(MODEL_PATH)
    )


    # --------------------------------------------------------
    # 12. ENTRENAMIENTO
    # --------------------------------------------------------

    try:

        results = model.train(

            # Dataset
            data=str(DATA_YAML),

            # Entrenamiento
            epochs=EPOCHS,

            imgsz=IMGSZ,

            # BATCH FIJO
            batch=BATCH,

            # GPU
            device=DEVICE,

            # Mixed Precision
            amp=AMP,

            # DataLoader
            workers=WORKERS,

            cache=CACHE,

            # OPTIMIZADOR CORRECTO
            optimizer="AdamW",

            # Reproducibilidad
            seed=SEED,

            deterministic=DETERMINISTIC,

            # Early stopping desactivado
            patience=PATIENCE,

            # Mosaic
            close_mosaic=CLOSE_MOSAIC,

            # Guardado
            save=True,

            save_period=SAVE_PERIOD,

            # Gráficas
            plots=PLOTS,

            # Salida
            project=str(
                ROOT / "runs"
            ),

            name=run_name,

            exist_ok=True,

            verbose=True,
        )


    except torch.cuda.OutOfMemoryError as exc:

        limpiar_cuda()

        raise RuntimeError(
            "\n" +
            "=" * 75 +
            "\n"
            "CUDA OUT OF MEMORY\n"
            "=" * 75 +
            "\n"
            "Configuración utilizada:\n"
            f"  Modelo : YOLO26X\n"
            f"  Batch  : {BATCH}\n"
            f"  ImgSz  : {IMGSZ}\n"
            f"  AMP    : {AMP}\n"
            "\n"
            "Para 8 GB VRAM prueba:\n"
            "  BATCH = 1\n"
            "  IMGSZ = 512\n"
            "  AMP = True\n"
            "=" * 75
        ) from exc


    # --------------------------------------------------------
    # 13. RESULTADOS
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("ENTRENAMIENTO TERMINADO")
    print("=" * 75)

    print(
        f"Resultados : {results.save_dir}"
    )

    print(
        "best.pt    : "
        f"{Path(results.save_dir) / 'weights' / 'best.pt'}"
    )

    print(
        "last.pt    : "
        f"{Path(results.save_dir) / 'weights' / 'last.pt'}"
    )

    print("=" * 75)


# ------------------------------------------------------------
# 14. EJECUCIÓN
# ------------------------------------------------------------

if __name__ == "__main__":
    main()


ACCESSAI - YOLO26X - ENTRENAMIENTO GPU
Ultralytics       : 8.4.69
PyTorch           : 2.14.0+cu126
CUDA PyTorch      : 12.6
CUDA disponible   : True
GPU               : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM libre/total  : 6.93 / 8.00 GB
Modelo            : C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\yolo26x.pt
Dataset           : C:\III DIPLOMA DE EXTENSIÓN UNIVERSITARIA EN INTELIGENCIA ARTIFICIAL AVANZADA SAMSUNG INNOVATION CAMPUS (2025-26)\proyecto2\DATA\ROD-Dataset\dataset\data_filtrado.yaml
Clases (4)        : ['Obstaculo_Dinamico', 'Obstaculo_Fijo', 'Barrera_Arquitectonica', 'Infraestructura_Peatonal']
Batch             : 1
Epochs            : 20
Image size        : 640
Device            : 0
AMP               : True
Optimizer         : AdamW
Workers           : 0 (Jupyter)
Cache             : disk
Patience          : 0
Close Mosaic      : 10
Preflight         : True

PREFLIGHT - 1 EPOCH / 2% DATASET

In [ ]:
best_model_path = sorted(Path('runs').rglob('best.pt'))[-1]
print('Mejor modelo:', best_model_path)

trained_model = YOLO(str(best_model_path))
metrics = trained_model.val(data=str(DATA_YAML), imgsz=640, split='test')
print(metrics)

## Gráficas y rediseño de análisis

Esta sección genera una vista compacta de las métricas principales del entrenamiento y guarda la gráfica para documentar el experimento.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plot_dir = ROOT / 'resultados'
plot_dir.mkdir(parents=True, exist_ok=True)

metrics_dict = getattr(metrics, 'results_dict', {})
if isinstance(metrics_dict, dict) and metrics_dict:
    keys_to_plot = []
    for key, value in metrics_dict.items():
        lower = key.lower()
        if any(token in lower for token in ['precision', 'recall', 'map', 'f1', 'loss']):
            keys_to_plot.append((key, float(value)))

    if keys_to_plot:
        labels = [k for k, _ in keys_to_plot]
        values = [v for _, v in keys_to_plot]

        plt.figure(figsize=(12, 6))
        bars = plt.bar(labels, values, color=['#4C78A8', '#F58518', '#54A24B', '#E45756', '#B279A2', '#72B7B2'])
        plt.title('Métricas principales de validación - AccessAI v5.2')
        plt.xticks(rotation=30, ha='right')
        plt.ylabel('Valor')
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        for bar, val in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        plt.tight_layout()
        save_plot = plot_dir / 'metricas_accessai_v5_2.png'
        plt.savefig(save_plot, dpi=180, bbox_inches='tight')
        print(f'Gráfica guardada en: {save_plot}')
        plt.show()
    else:
        print('No se encontraron métricas numéricas para graficar en results_dict.')
else:
    print('El objeto metrics no expone resultados numéricos para crear la gráfica.')

In [ ]:
import json
from pathlib import Path

run_dirs = sorted(Path('runs').glob('AccessAI_YOLO26X_4clases_*'))
if run_dirs:
    latest_run = run_dirs[-1]
    log_file = latest_run / 'results.csv'
    print('Directorio de entrenamiento:', latest_run)
    print('Archivo CSV:', log_file)

    if log_file.exists():
        import pandas as pd
        df = pd.read_csv(log_file)
        if 'epoch' in df.columns and any(col.lower().endswith('loss') for col in df.columns):
            loss_cols = [col for col in df.columns if col.lower().endswith('loss')]
            plt.figure(figsize=(12, 6))
            for col in loss_cols[:3]:
                plt.plot(df['epoch'], df[col], label=col)
            plt.title('Pérdida por epoch - AccessAI v5.2')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            loss_plot = plot_dir / 'loss_por_epoch_accessai_v5_2.png'
            plt.savefig(loss_plot, dpi=180, bbox_inches='tight')
            print(f'Gráfica de pérdidas guardada en: {loss_plot}')
            plt.show()
        else:
            print('No se encontraron columnas de pérdida en results.csv.')
    else:
        print('No existe results.csv en la última ejecución. Ejecuta primero el entrenamiento.')
else:
    print('No se encontraron ejecuciones en la carpeta runs/.')

## 2. Evolución del mAP por epoch

Esta gráfica compara el progreso de la precisión media por epoch, útil para diagnosticar convergencia del modelo.

In [ ]:
run_dirs = sorted(Path('runs').glob('AccessAI_YOLO26X_4clases_*'))
if run_dirs:
    latest_run = run_dirs[-1]
    log_file = latest_run / 'results.csv'
    if log_file.exists():
        import pandas as pd
        df = pd.read_csv(log_file)
        map_candidates = [c for c in df.columns if 'map' in c.lower() or 'mAP' in c]
        map_candidates = [c for c in map_candidates if '50' in c or '95' in c or 'map' in c.lower()]
        if 'epoch' in df.columns and map_candidates:
            plt.figure(figsize=(12, 6))
            for col in map_candidates[:5]:
                plt.plot(df['epoch'], df[col], label=col)
            plt.title('mAP por epoch - AccessAI v5.2')
            plt.xlabel('Epoch')
            plt.ylabel('mAP')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            map_plot = plot_dir / 'map_por_epoch_accessai_v5_2.png'
            plt.savefig(map_plot, dpi=180, bbox_inches='tight')
            print(f'Gráfica de mAP guardada en: {map_plot}')
            plt.show()
        else:
            print('No se encontraron columnas de mAP en results.csv.')
    else:
        print('No existe results.csv para graficar el mAP.')
else:
    print('No se encontraron ejecuciones previas para analizar mAP.')

## 3. Resumen final de métricas

Este bloque presenta un resumen compacto con los principales indicadores del modelo y la ruta del mejor checkpoint.

In [ ]:
from pathlib import Path

best_candidates = sorted(Path('runs').rglob('best.pt'))
if best_candidates:
    best_model = best_candidates[-1]
    print('Mejor checkpoint:', best_model)

    if 'metrics' in globals():
        print('\nMétricas observadas:')
        print(metrics)
    else:
        print('El objeto metrics no está definido en este entorno.')
else:
    print('Todavía no existe un checkpoint best.pt en runs/.')

## Inferencia sobre una imagen

Selecciona una imagen en la carpeta de prueba o un archivo nuevo para comprobar la salida del modelo.

In [ ]:
image_path = ROOT / 'tests' / 'imgs' / 'sample.jpg'
output_dir = ROOT / 'resultados' / 'inferencia_accessai_v5_2'

if not image_path.exists():
    print('No se encontró la imagen de prueba por defecto:', image_path)
    print('Puedes definir otra ruta como image_path antes de ejecutar la celda.')
else:
    results = trained_model.predict(
        source=str(image_path),
        conf=0.25,
        save=True,
        project=str(output_dir.parent),
        name=output_dir.name,
        exist_ok=True,
    )
    print('Resultado de inferencia:', results)
    print('Directorio de salida:', output_dir)

## Nota final

Este notebook representa una versión experimental del prototipo. La detección por clases agrupadas sirve para validar el flujo del pipeline, pero no sustituye una validación real de accesibilidad urbana con un dataset específico y métricas de prueba, como `precision`, `recall`, `F1` y `mAP`.